# CSCI 443 Homework 4

**Due Date: Thursday, March 26**

In homework 3, we left off with Z-scores and sampling distributions.  

In this homework we focus on confidence intervals and the effects of
skewness on computing confidence intervals.

**Important Note to Students:**

This homework is designed to help you develop intuition about confidence intervals, sampling distributions, and the behavior of statistical estimators with different sample sizes. These concepts are foundational to understanding statistical inference and quality control applications.

**Please approach these problems without using AI assistance initially.** The purpose of this assignment is to build your understanding through hands-on calculation, coding, and reflection. While AI tools can be helpful for debugging syntax errors or looking up function signatures, over-reliance on AI to solve the problems will defeat the learning objectives and leave you without the intuition needed for exams and real-world applications.

Work through the problems step-by-step, experiment with the code, observe the results, and think critically about what the numbers are telling you. Your future self will thank you for taking the time to truly understand these concepts now.

   
## Part 1: Gaussian Confidence Intervals. 

Assume we have experience with the underlying processes that generate samples and know that the underlying distribution is Gaussian or nearly Gaussian.

**Problem 1.1** You draw the samples in sample set S from this distribution.  Compute a 99% confidence interval using Z scores. 

$$S = \\{5, 3, 9, 2, 5\\}$$



In [0]:

# USEFUL CODES WITH SCIPY TO CREATE CODE.
# from scipy.stats import norm


# norm.pdf(x, loc=μ, scale=σ)
# → Probability Density Function
# → "How dense/likely is value x?"
# → Example: norm.pdf(0, 0, 1)

# norm.cdf(x, loc=μ, scale=σ)
# → Cumulative Distribution Function
# → "Probability that X ≤ x"
# → Example: norm.cdf(1.96) ≈ 0.975

# norm.ppf(p, loc=μ, scale=σ)
# → Percent Point Function (inverse CDF)
# → "Value x such that P(X ≤ x) = p"
# → Used for Z-scores and confidence intervals
# → Example: norm.ppf(0.975) ≈ 1.96

# norm.rvs(loc=μ, scale=σ, size=n)
# → Generate random samples from a normal distribution
# → Example: norm.rvs(0, 1, size=10)

# norm.interval(confidence, loc=μ, scale=σ)
# → Returns confidence interval directly
# → Example: norm.interval(0.99, loc=mean, scale=std/√n)

# norm.mean(loc=μ, scale=σ)
# → Returns mean (μ)

# norm.std(loc=μ, scale=σ)
# → Returns standard deviation (σ)

# --- QUICK INTUITION ---
# cdf(x) → "How much is to the LEFT of x?"
# ppf(p) → "Where is the cutoff for probability p?"
# pdf(x) → "How tall is the curve at x?"

# --- COMMON VALUES ---
# norm.ppf(0.90)  ≈ 1.28
# norm.ppf(0.95)  ≈ 1.645
# norm.ppf(0.975) ≈ 1.96   # 95% CI
# norm.ppf(0.995) ≈ 2.576  # 99% CI



In [0]:
import numpy as np
from scipy.stats import norm


S = np.array([5, 3, 9, 2, 5])

mean = np.mean(S)
std = np.std(S, ddof=1)
n = len(S)

z = norm.ppf(0.995 )  

margin = z * (std / np.sqrt(n))

ci_lower = mean - margin
ci_upper = mean + margin



print(mean)
print(std)
print(ci_lower,ci_upper)

**Problem 1.2** Write Python code to compute the standard deviation of a sample set using the inverse CDF (quantile function) of the Gaussian distribution. You may use scipy or numpy, but do not use any function that directly computes the confidence interval.  Use this code to confirm the result to problem 1.1.

In [0]:
import numpy as np
from scipy.stats import norm

S = np.array([5, 3, 9, 2, 5])
n = len(S)

mean = np.mean(S)
variance = np.sum((S - mean) ** 2) / (n - 1)
std = np.sqrt(variance)

confidence = 0.99
alpha = 1 - confidence
z = norm.ppf(1 - alpha / 2)   # ≈ 2.576 for 99%


margin = z * (std / np.sqrt(n))
ci_lower = mean - margin
ci_upper = mean + margin

std_np = np.std(S, ddof=1)

print(S)
print(n)
print(mean)
print(std)
print(std_np)
print(z)
print(ci_lower,ci_upper)


**Problem 1.3** You built live instrumentation for a production line at a refinery.  For each batch of a refined material, live instrumentation measures n=5 samples, generates a confidence interval, and reports the sample mean of the amount of impurities. To validate the system, you collect a large dataset and divide it into groups of 5 samples. Using all samples in the file `prob1.csv`, plot a histogram of the sampling distribution (i.e., the distribution of sample means from each group of 5).

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob1.csv', header=0).iloc[:,0].values

n = 5
num_groups = len(S) // n
sample_means = [np.mean(S[i*n:(i+1)*n]) for i in range(num_groups)]

plt.hist(sample_means, bins=30, edgecolor="black")
plt.xlabel("Sample Mean")
plt.ylabel("Frequency")
plt.title("Sampling Distribution of Sample Means (n=5)")
plt.show()

**Problem 1.4** Using the full dataset in `prob1.csv`, randomly sample 1000 groups of size n=5 (with replacement). For each group, compute a 99% confidence interval for the mean using a Gaussian approximation (i.e., use the Z-score as in Problems 1.1 and 1.2).

Next, calculate the population mean from the entire dataset in `prob1.csv`.

What fraction of the 1000 confidence intervals contain the true population mean?

ANSWER: it would land between 0.92-0.97 

Reflect: Is this fraction approximately 99%, as expected from the confidence level? Explain why or why not.



In [0]:
import numpy as np
import pandas as pd
from scipy.stats import norm

S_full = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob1.csv', header=0).iloc[:,0].values

n = 5
num_samples = 1000
confidence = 0.99
alpha = 1 - confidence
z = norm.ppf(1 - alpha / 2) 


pop_mean = np.mean(S_full)
intervals = []
for _ in range(num_samples):
    sample = np.random.choice(S_full, size=n, replace=True)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    margin = z * (sample_std / np.sqrt(n))
    ci_lower = sample_mean - margin
    ci_upper = sample_mean + margin
    intervals.append((ci_lower, ci_upper))

contains_mean = [ci_lower <= pop_mean <= ci_upper for ci_lower, ci_upper in intervals]
fraction = np.mean(contains_mean)


print(pop_mean)
print(fraction)

**Problem 1.5** Repeat Problem 1.4, but this time use n=35 samples per group instead of n=5. Randomly sample 1000 groups of size n=35 (with replacement) from `prob1.csv`. For each group, compute a 99% confidence interval for the mean using a Gaussian approximation.

Compute what fraction of the 1000 confidence intervals contain the true population mean.

Reflect: How does this fraction compare to the result from Problem 1.4 (which used n=5)? Why does increasing the sample size improve the coverage rate?

In [0]:
import numpy as np
import pandas as pd
from scipy.stats import norm

S_full = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob1.csv', header=0).iloc[:,0].values

n = 35
num_samples = 1000
confidence = 0.99
alpha = 1 - confidence
z = norm.ppf(1 - alpha / 2)

pop_mean = np.mean(S_full)
intervals = []
for _ in range(num_samples):
    sample = np.random.choice(S_full, size=n, replace=True)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    margin = z * (sample_std / np.sqrt(n))
    ci_lower = sample_mean - margin
    ci_upper = sample_mean + margin
    intervals.append((ci_lower, ci_upper))

contains_mean = [ci_lower <= pop_mean <= ci_upper for ci_lower, ci_upper in intervals]
fraction = np.mean(contains_mean)

print(pop_mean)
print(fraction)

**Problem 1.6**  
Suppose we want our quality control confidence intervals to be empirically accurate: specifically, we want at least 98% of our 99% confidence intervals to contain the true population mean. For the dataset in `prob1.csv`, determine the minimum sample size `n` (for groups of size `n` drawn with replacement) for which the fraction of 10000 Gaussian 99% confidence intervals containing the true is within 1% of the 99% target confidence (i.e., exceeds 98%).

Present your code, plot the coverage as a function of `n`, and report the minimum `n` that satisfies the requirement.  
Briefly discuss the practical trade-off between statistical confidence and sampling cost.

In [0]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt


S_full = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob1.csv', header=0).iloc[:,0].values

pop_mean = np.mean(S_full)
confidence = 0.99
alpha = 1 - confidence
z = norm.ppf(1 - alpha / 2)
num_samples = 10000

coverages = []
n_values = range(2, 51)  # Try n from 2 to 50

for n in n_values:
    intervals = []
    for _ in range(num_samples):
        sample = np.random.choice(S_full, size=n, replace=True)
        sample_mean = np.mean(sample)
        sample_std = np.std(sample, ddof=1)
        margin = z * (sample_std / np.sqrt(n))
        ci_lower = sample_mean - margin
        ci_upper = sample_mean + margin
        intervals.append((ci_lower, ci_upper))
    contains_mean = [ci_lower <= pop_mean <= ci_upper for ci_lower, ci_upper in intervals]
    coverage = np.mean(contains_mean)
    coverages.append(coverage)

min_n = next(n for n, cov in zip(n_values, coverages) if cov >= 0.98)

plt.plot(n_values, coverages, marker='o')
plt.axhline(0.98, color='red', linestyle='--', label='98% threshold')
plt.xlabel('Sample Size n')
plt.ylabel('Empirical Coverage')
plt.title('Empirical Coverage of 99% CI vs Sample Size')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Minimum sample size n for empirical coverage > 98%:", min_n)
print("Trade-off: Higher n increases confidence but also sampling cos Smaller n may not achieve desired coverage.")

**Problem 1.7**  
A coworker reviews your analysis from Problems 1.4-1.6 and points out that the Gaussian approximation may not be appropriate for small sample sizes. She suggests using the **Student's t-distribution** instead, which accounts for the additional uncertainty introduced when estimating the population standard deviation from small samples.

The t-distribution has heavier tails than the normal distribution, producing wider confidence intervals that better reflect the uncertainty in small-sample estimates. As sample size increases, the t-distribution converges to the normal distribution.

Repeat Problem 1.4 using the **t-distribution** instead of the Gaussian (Z-score) approach:
* Randomly sample 1000 groups of size n=5 from `prob1.csv`
* For each group, compute a 99% confidence interval using the t-distribution with appropriate degrees of freedom
* Calculate what fraction of these intervals contain the true population mean

Compare your result to Problem 1.4 (which used the Gaussian approach with n=5 and achieved ~94% coverage). Does the t-distribution produce confidence intervals with coverage closer to the nominal 99%?

ANSWER: The t-distribution makes bigger, wider guesses because it knows it doesn't have enough information, so it plays it safe and gets the right answer more often

**Hint**: Use `scipy.stats.t.ppf()` to get the critical t-value, with degrees of freedom = n-1.

In [0]:
import numpy as np
import pandas as pd
from scipy.stats import t

S_full = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob1.csv', header=0).iloc[:,0].values

n = 5
num_samples = 1000
confidence = 0.99
alpha = 1 - confidence
df = n - 1
t_crit = t.ppf(1 - alpha / 2, df)

pop_mean = np.mean(S_full)
intervals = []
for _ in range(num_samples):
    sample = np.random.choice(S_full, size=n, replace=True)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    margin = t_crit * (sample_std / np.sqrt(n))
    ci_lower = sample_mean - margin
    ci_upper = sample_mean + margin
    intervals.append((ci_lower, ci_upper))

contains_mean = [ci_lower <= pop_mean <= ci_upper for ci_lower, ci_upper in intervals]
fraction = np.mean(contains_mean)

print(pop_mean)
print(fraction)

## Part 2: Formal definition for Skewness

We mostly covered skewness empirically.  "Follow the tails" being a useful mnemonic.

In most cases, a left (negative) skewed distribution is characterized by having
a mean that is less than the median, i.e., the heavy tail on the left has pulled
the mean to the left of the median.  Conversly, a right skewed distribuion is 
characterized by having a mean that is greater than the median.  We will not
encounter any distributions in this class where this is not true.  However, 
the relationship between the mean and median is merely a consequence, it isn't
the formal definition of skewness.  Skewness (\\(\gamma\\)) is most often defined 
as follows

$$\text{Skewness } \gamma = \frac{E[(X-\mu)^3]}{\sigma^3}$$

The sample variance \\(\sigma^2= \frac{1}{n-1}\sum (x_i-\bar{x})^2\\)
uses \\(\frac{1}{n-1}\\) instead of \\(\frac{1}{n}\\) to 
account for sample variance being computed from the same samples
as the sample mean.  

Sample Skewness (\\(G_1\\)) has a somewhat more complicated correction
to handle the bias introduced by computing the sample mean, sample
standard deviation, and sample skewness from the same samples.

$$G_1 = \frac{n}{(n-1)(n-2)}\sum_{i=1}^n \bigg(\frac{x_i-\bar{x}}{s}\bigg)^3$$

where

  - \\(n\\) is the number of samples
  - \\(x_i\\) is the \\(i^{th}\\) sample.
  - \\(\bar{x}\\) is the sample mean.

\\(\frac{n}{(n-1)(n-2)}\\) includes the correction for the bias 
introduced by computing \\(\bar{x}\\), \\(s\\), and 
\\(G_1\\) from the same samples.  The derivation of this
correction is outside the scope of the course, but I welcome
the student to study it further should they wish.




**Problem 2.1** Create code in this notebook to compute the sample skewness.


In [0]:
import numpy as np
import pandas as pd
from scipy.stats import t

def sample_skewness(samples) -> float:
    n = len(samples)
    mean = np.mean(samples)
    std = np.std(samples, ddof=1)
    skew = (n / ((n - 1) * (n - 2))) * np.sum(((samples - mean) / std) ** 3)
    return skew


S = np.array([5, 3, 9, 2, 5])
print(sample_skewness(S))

**Problem 2.2** Write a unit test in this notebook that confirms that your
implementation of sample skewness returns near 0 \\(\pm 0.05\\) for a
sufficient number of samples drawn from U[0,1].  U[0,1] is symmetric.
All symmetric distributions exhibit 0 skewness.  


In [0]:
def sample_skewness(samples):
    n = len(samples)
    mean = np.mean(samples)
    std = np.std(samples, ddof=1)
    return np.sum((samples - mean)**3) / ((n - 1) * std**3)

np.random.seed(42)
samples = np.random.uniform(0, 1, 1000)
skew = sample_skewness(samples)
assert abs(skew) < 0.05, f"Skewness {skew} not within ±0.05"
print("Unit test passesd: sample skewness for U 0 , 1 is within +-0.05.")

   
**Problem 2.3** Write a second unit test that confirms your implementation correctly detects positive skewness. The exponential distribution with \\(\lambda=1\\) is right-skewed and has a theoretical skewness of 2. Draw a sufficient number of samples from an exponential random variable with \\(\lambda=1\\) and verify that your `sample_skewness` function returns a value within \\(2 \pm 0.05\\), confirming that the distribution exhibits the expected positive (right) skewness.

In [0]:
def sample_skewness(samples):
    n = len(samples)
    mean = np.mean(samples)
    std = np.std(samples, ddof=1)
    return np.sum((samples - mean)**3) / ((n - 1) * std**3)

np.random.seed(42)
samples = np.random.exponential(scale=1, size=1000)
skew = sample_skewness(samples)
if abs(skew - 2) < 0.05:
    print("Unit test passed: sample skewness for exponential is within 2±0.05.")
else:
    print("Unit test failed: skewness =", skew)

**Problem 2.4** Skewness of a distribution is defined as

<!--
\\[ \xcancel{E[(X - \mu)^3] = \int_{-\infty}^{\infty} (x - \mu)^3 f(x) \, dx} \\]

*Revised*
-->

\\[ \frac{E[(X - \mu)^3]}{\sigma^3} = \int_{-\infty}^{\infty} \frac{(x - \mu)^3}{\sigma^3} f(x) \, dx \\]

A triangular distribution is defined by three parameters.  It has the pdf

\\[
f(x) = \begin{cases} 
  0      & \text{if } x \leq a \\
  \frac{2\cdot(x-a)}{(b-a)(c-a)}& \text{if } a \leq x \leq c \\
  \frac{2\cdot(c-x)}{(b-a)(c-b)}& \text{if } c \leq x \leq b \\
  0 & \text{if } x \geq b \\
\end{cases}
\\]

where 

 * \\(a\\) is the left corner of the triangle, i.e., the lower bound
   of the distribution's pdf.   
 * \\(b\\) is the right corner of the triangle, i.e., the upper bound
   of the distribution's pdf.
 * \\(c\\) is at the peak of the triangle, i.e., the mode of the
   distribution's pdf.

An example triangular distribution is shown below.


In [0]:
import numpy as np
import matplotlib.pyplot as plt

def plot_triangular_distribution(a, b, c):
    # Generate x values across the support
    x = np.linspace(a, b, 500)
    y = np.zeros_like(x)
    
    # Compute the pdf according to the triangular definition
    for i, xi in enumerate(x):
        if a <= xi <= c:
            y[i] = 2 * (xi - a) / ((b - a) * (c - a))
        elif c < xi <= b:
            y[i] = 2 * (b - xi) / ((b - a) * (b - c))
        # else remain 0
    
    plt.plot(x, y, lw=2, color='navy')
    plt.fill_between(x, y, color='skyblue', alpha=0.5)
    plt.xlabel('x')
    plt.ylabel('Density')
    plt.title(f'Triangular Distribution PDF\n a={a}, b={b}, c={c}')
    plt.grid(True, alpha=0.3)
    plt.show()

In [0]:
plot_triangular_distribution(a=0, b=1, c=2/3)

The mean of a triangular distribution is given by 

\\[ \mu = \frac{a + b + c}{3}\\]

A right triangular distribution with the peak on the right edge of the 
triangle has \\(b=c\\), as shown below.  The triangle is shifted to the 
left so that the mean resides at 0.


PDF of a right triangle distribution:

\\[ f(x) = \frac{2(x-a)}{(b-a)^2} \quad \text{for } a \leq x \leq b \\]

However, \\(b-a=1\\), \\(a=-2/3\\), and \\(b=1/3\\) causing the above to simplify

\\[ f(x) = 2(x+\frac{2}{3}) \quad \text{for } -\frac{2}{3} \leq x \leq \frac{1}{3} \\]

Derive the skewness for this distribution.

 ## Problem 2.4 — Skewness of Right Triangular Distribution

  **Given:** The distribution is shifted so that $\mu = 0$, with pdf:

  $$f(x) = 2\left(x + \frac{2}{3}\right), \quad -\frac{2}{3} \leq x \leq \frac{1}{3}$$

  The skewness formula reduces to:

  $$\text{skewness} = \frac{E[X^3]}{\sigma^3}$$

  ---

  ### Step 1 — Compute $\sigma^2$

  Since $\mu = 0$, $\sigma^2 = E[X^2]$:

  $$\sigma^2 = \int_{-2/3}^{1/3} x^2 \cdot 2\left(x + \frac{2}{3}\right)\, dx = 2\int_{-2/3}^{1/3} \left(x^3 +
  \frac{2}{3}x^2\right) dx$$

  $$= 2\left[\frac{x^4}{4} + \frac{2x^3}{9}\right]_{-2/3}^{1/3}$$

  At $x = \frac{1}{3}$:

  $$\frac{1}{324} + \frac{2}{243} = \frac{3}{972} + \frac{8}{972} = \frac{11}{972}$$

  At $x = -\frac{2}{3}$:

  $$\frac{16}{324} - \frac{16}{243} = \frac{48}{972} - \frac{64}{972} = -\frac{16}{972}$$

  $$\sigma^2 = 2 \cdot \left(\frac{11}{972} - \left(-\frac{16}{972}\right)\right) = 2 \cdot \frac{27}{972} = 2 \cdot
  \frac{1}{36} = \frac{1}{18}$$

  $$\sigma = \frac{1}{\sqrt{18}}, \qquad \sigma^3 = \frac{1}{18\sqrt{18}} = \frac{1}{54\sqrt{2}}$$

  ---

  ### Step 2 — Compute $E[X^3]$

  $$E[X^3] = \int_{-2/3}^{1/3} x^3 \cdot 2\left(x + \frac{2}{3}\right) dx = 2\int_{-2/3}^{1/3} \left(x^4 +
  \frac{2}{3}x^3\right) dx$$

  $$= 2\left[\frac{x^5}{5} + \frac{x^4}{6}\right]_{-2/3}^{1/3}$$

  At $x = \frac{1}{3}$:

  $$\frac{1}{1215} + \frac{1}{486} = \frac{2}{2430} + \frac{5}{2430} = \frac{7}{2430}$$

  At $x = -\frac{2}{3}$:

  $$-\frac{32}{1215} + \frac{16}{486} = -\frac{64}{2430} + \frac{80}{2430} = \frac{16}{2430}$$

  $$E[X^3] = 2 \cdot \left(\frac{7}{2430} - \frac{16}{2430}\right) = 2 \cdot \left(-\frac{9}{2430}\right) =
  -\frac{1}{135}$$

  ---

  ### Step 3 — Compute Skewness

  $$\text{skewness} = \frac{E[X^3]}{\sigma^3} = \frac{-\frac{1}{135}}{\frac{1}{54\sqrt{2}}} = -\frac{54\sqrt{2}}{135} =
  \boxed{-\frac{2\sqrt{2}}{5} \approx -0.566}$$

  The negative value confirms **left skew** 
 

## Part 3: Some intuition behind skewness

With variance we look at the sum of the squares 
\\(\sigma^2= \frac{1}{n-1}\sum (x_i-\bar{x})^2\\).
The square means that all terms in the summation are positive.  The 
further samples are away from the mean in either direction their
impact on the variance increases with the square of that distance.
Skewness uses the sum of the cubes.  Consider the case when
\\(\bar{x}\\) is 0 and let \\(C\\) denote \\(n/((n-1)(n-2))\\). The
computation of sample skewness simplifes to a sum of cubics: 

$$C\cdot \sum_{i=0}^{n} x^3$$

\\(x^3\\) is an odd function and as such any sample which is to the left of the
mean by \\(d_i = x_i-\bar{x}\\) will cancel with a sample the same distance on the
right side \\(d_j = x_j-\bar{x}\\) if \\(d_i = -d_j\\).  A symetric distribution 
thus has 0 skewness regardles of its variance.


**Problem 3.1** Write code in this notebook to draw 100 samples from a uniform random
variable \\(U[0,1]\\). Use matplotlib to create a relative
frequency histogram samples with 5 bins to confirm that the shape is 
roughly uniform.  Not all buckets will obtain the same number of samples.   
Aside: We will go over tests for measuring how close samples match a 
hypothesized distribution, for now a visual assessment is adequate.


In [0]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
samples = np.random.uniform(0, 1, 100)

plt.hist(samples, bins=5, edgecolor='black', density=True)
plt.xlabel('Value')
plt.ylabel('Relative Frequency')
plt.title('Histogram of 100 Samples from U[0,1]')
plt.grid(True, alpha=0.3)
plt.show()

**Problem 3.2** Create a plot that shows the sum of the squares 
\\(\sum_{i=1}^n (x_i-\bar{x})^2 \\) for the same samples as generated in Problem 3.1.
The x-axis should be \\(i\\) where \\(i\\) denotes the \\(i^{th}\\) sample.
The y-axis should be the sum of the squares up to the \\(i^{th}\\) sample.
This plot should be non-decreasing.  It demonstrates how sum of squares increases
whether samples fall above or below the mean.  

In [0]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
samples = np.random.uniform(0, 1, 100)
mean = np.mean(samples)

sq_diffs = (samples - mean) ** 2
cum_sum_squares = np.cumsum(sq_diffs)

plt.plot(np.arange(1, len(samples) + 1), cum_sum_squares, marker='o', linestyle='-')
plt.xlabel('Sample Index i')
plt.ylabel('Cumulative Sum of Squares up to i')
plt.title('Cumulative Sum of Squares for U[0,1] Samples')
plt.grid(True, alpha=0.3)
plt.show()

**Problem 3.3**  Order the samples from Problem 3.1 from smallest to largest and create another plot.  


In [0]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
samples = np.random.uniform(0, 1, 100)

sorted_samples = np.sort(samples)

plt.plot(sorted_samples, marker='o', markersize=3, linewidth=1, color='steelblue')
plt.xlabel('Rank (index)')
plt.ylabel('Value')
plt.title('Sorted Samples from U[0,1]')
plt.grid(True, alpha=0.3)
plt.show()

**Problem 3.4** When the samples are ordered, what happens? Explain the shape. What does the shape of the curve tell us about the contribution to the variance of points that are farther from the mean?

When sorted, the samples form a straight line because uniform data rises at a constant rate, and points far from the mean (the ends) contribute the most to variance since their squared deviations are largest.

**Problem 3.5** Create another plot that shows the sum of the 
cubes \\(\sum_{i=1}^n (x_i - \bar{x})^3\\) for the same samples 
generated in Problem 3.1.  The resulting plot should NOT be
increasing.

In [0]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
samples = np.random.uniform(0, 1, 100)
mean = np.mean(samples)

cube_diffs = (samples - mean) ** 3
cum_sum_cubes = np.cumsum(cube_diffs)

plt.plot(np.arange(1, len(samples) + 1), cum_sum_cubes, marker='o', linestyle='-')
plt.xlabel('Sample Index i')
plt.ylabel('Cumulative Sum of Cubes up to i')
plt.title('Cumulative Sum of Cubes for U[0,1] Samples')
plt.grid(True, alpha=0.3)
plt.show()

**Problem 3.6** Order the samples from smallest to largest and create the same plot again
but with the ordered samples.  


In [0]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
samples = np.random.uniform(0, 1, 100)
mean = np.mean(samples)

sorted_samples = np.sort(samples)
cube_diffs = (sorted_samples - mean) ** 3
cum_sum_cubes = np.cumsum(cube_diffs)

plt.plot(np.arange(1, len(sorted_samples) + 1), cum_sum_cubes, marker='o', linestyle='-')
plt.xlabel('Sample Index (Ordered)')
plt.ylabel('Cumulative Sum of Cubes up to i')
plt.title('Cumulative Sum of Cubes for Ordered U[0,1] Samples')
plt.grid(True, alpha=0.3)
plt.show()

**Problem 3.7** How does this plot differ from the plot for cumulative sum of squares? Note that the function is no longer non-decreasing. How does the shape of the curve affect the contribution of the samples to the left of the mean vs. to the right of the mean for an unskewed distribution


Unlike squares which always add positively, cubes preserve sign so points left of the mean pull the sum down and points right pull it up, and for a symmetric distribution they cancel out leaving the total near zero.

**Problem 3.8** How does the shape of this curve affect the contribution of samples 
farther from the mean than nearer to the mean? What happens if more samples
are near the mean on one-side of the distribution as would occur with 
a skewed distribution?

Cubing is a superlinear amplifier a point twice as far from the mean contributes 8x more to the skewness sum, so the tails dominate the calculation entirely; in a skewed distribution the asymmetric tail has more extreme deviations that go uncanceled, driving the skewness away from zero.

## Part 4: Confidence Intervals for Instrument Data Obeying an Unknown Disribution

You are designing instrumentation for a physical system and the collected data does not follow a known distribution. The sample data is provided in the file `prob4.csv` in the same directory as this notebook.



**Problem 4.1**
Load the data from `prob4.csv` and plot a histogram of its values. Carefully observe the shape of the distribution. Is it symmetric or skewed? Record your observations and discuss any features that stand out.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob4.csv', header=0).iloc[:,0].values

plt.hist(S, bins=30, edgecolor='black', density=True)
plt.xlabel('Value')
plt.ylabel('Relative Frequency')
plt.title('Histogram of prob4.csv Values')
plt.grid(True, alpha=0.3)
plt.show()

is skwed to the right

**Problem 4.2**
Compute the skewness of the data in `prob4.csv` using your `sample_skewness` function from earlier in this homework. Report the computed skewness and comment on what it indicates about the shape of the distribution.

In [0]:
import numpy as np
import pandas as pd

def sample_skewness(samples):
    n = len(samples)
    mean = np.mean(samples)
    std = np.std(samples, ddof=1)
    return np.sum((samples - mean)**3) / ((n - 1) * std**3)

S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob4.csv', header=0).iloc[:,0].values
skew = sample_skewness(S)
print(f"skewnes: {skew:.4f}")
if skew > 0:
    print("The distribution is rightskewed")
elif skew < 0:
    print("The distribution is leftskewed")
else:
    print("The distribution is symmetric.")

**Problem 4.3**
For n=4, randomly sample 10000 groups from the data in `prob4.csv`. Calculate a 99% confidence interval for the mean of each group using the t-distribution. What fraction of the intervals contain the true population mean? How many produce impossible (negative) interval values for the mean?

In [0]:
import numpy as np
import pandas as pd
from scipy.stats import t

# Load data
S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob4.csv', header=0).iloc[:,0].values

n = 4
num_groups = 10000
alpha = 0.01
t_crit = t.ppf(1 - alpha/2, df=n-1)
pop_mean = np.mean(S)

contains_mean = 0
negative_interval = 0

for _ in range(num_groups):
    group = np.random.choice(S, n, replace=False)
    group_mean = np.mean(group)
    group_std = np.std(group, ddof=1)
    margin = t_crit * group_std / np.sqrt(n)
    ci_lower = group_mean - margin
    ci_upper = group_mean + margin
    if ci_lower <= pop_mean <= ci_upper:
        contains_mean += 1
    if ci_lower < 0:
        negative_interval += 1

fraction_contain = contains_mean / num_groups

print(f"Fracion of intervals containing the true mean {fraction_contain:.4f}")
print(f"Number of intervals with negative lower bound {negative_interval}")

What fraction of the intervals contain the true population mean? 

ANSWER Coverage falls well below 99% due to heavy skew.

How many produce impossible (negative) interval values for the mean?
ANSWER Many lower bounds go negative because t-critic times std is huge.


**Problem 4.4**
Discuss: Based on your results, does the t-distribution approach provide reliable coverage and sensible interval values for this data? Why or why not?  What percentage of intervals contain the sample mean?

No, because the data is lopsided and the tdistribution expects symmetric data, so the intervals are wrong they miss the true mean too often and sometimes go negative, which is impossible for impurity measurements.

**Problem 4.5**
Use bootstrapping to empirically determine 99% confidence intervals for the sample mean (with n=4 samples per group). Generate 10000 confidence intervals using this approach.  What percentage contain the sample mean?  Compare the bootstrap intervals to those produced by the t-distribution.

Bootstrap intervals will be more sensible no negative bounds and slightly closer to 99% coverage, but both methods underperfoorm because n=4 is too small to reliably capture a heavily skewed distributionn

In [0]:
import numpy as np
import pandas as pd


S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob4.csv', header=0).iloc[:,0].values

n = 4
num_groups = 10000
alpha = 0.01
pop_mean = np.mean(S)

contains_mean = 0
negative_interval = 0

for _ in range(num_groups):
    group = np.random.choice(S, n, replace=False)
    boot_means = [np.mean(np.random.choice(group, n, replace=True)) for _ in range(1000)]
    ci_lower, ci_upper = np.percentile(boot_means, [alpha/2*100, (1-alpha/2)*100])
    if ci_lower <= pop_mean <= ci_upper:
        contains_mean += 1
    if ci_lower < 0:
        negative_interval += 1

fraction_contain = contains_mean / num_groups

print(f"Fraction of bootstrap intervals containing the true mean: {fraction_contain:.4f}")
print(f"Number of bootstrap intervals with negative lower bound: {negative_interval}")

**Problem 4.6**

Monte Carlo simulation is a computational technique used to approximate the properties of a distribution by randomly sampling from observed data. In this context, you will use Monte Carlo simulation to generate an empirical sampling distribution for the sample mean of n=4 for the instrument data in prob4.csv. 

- Randomly sample many groups of n=4 from the dataset.
- Compute the mean for each group.
- Plot a histogram of the resulting distribution of sample means.

How could you use this empirical sampling distribution to construct confidence intervals for the sample mean?

**Problem 4.7**
Now, use your empirical distribution (from Problem 4.6) to construct 10,000 confidence intervals for n=4. For example, you may use the 0.5th and 99.5th percentiles of the empirical distribution as the interval bounds. Compute what percentage of these intervals contain the true population mean of the entire dataset, and how many intervals have a negative lower end. Compare your results to the intervals constructed using the t-distribution in Problem 4.3.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob4.csv', header=0).iloc[:,0].values

n = 4
num_groups = 10000
pop_mean = np.mean(S)

# Problem 4.6 - Sample means
sample_means = [np.mean(np.random.choice(S, n, replace=False)) for _ in range(num_groups)]

plt.hist(sample_means, bins=30, edgecolor='black', density=True)
plt.xlabel('Sample Mean')
plt.ylabel('Relative Frequency')
plt.title('Sampling Distribution of Sample Mean (n=4)')
plt.grid(True, alpha=0.3)
plt.show()

# Problem 4.7 - Empirical CI
low, high = np.percentile(sample_means, 0.5), np.percentile(sample_means, 99.5)
print(f"Empirical CI:            ({low:.4f}, {high:.4f})")
print(f"Empirical coverage:      {(low <= pop_mean <= high):.4f}")
print(f"Empirical negative CIs:  {int(low < 0)}")

# t-distribution CI
t_crit = t.ppf(0.995, df=n-1)
t_contains, t_neg = 0, 0
for _ in range(num_groups):
    g = np.random.choice(S, n, replace=False)
    m, s = np.mean(g), np.std(g, ddof=1)
    lo, hi = m - t_crit*s/np.sqrt(n), m + t_crit*s/np.sqrt(n)
    t_contains += lo <= pop_mean <= hi
    t_neg += lo < 0

print(f"T-distribution coverage: {t_contains/num_groups:.4f}")
print(f"T-distribution neg CIs:  {t_neg}")

**Problem 4.8**
Suppose, due to cost constraints, you are only able to collect the first 100 data points from `prob4.csv` (rather than the full dataset). Using classical bootstrapping:

- Draw 10,000 bootstrap samples, each of size n=100 (sampled with replacement from these 100 points).
- For each bootstrap sample, compute the sample mean.
- Plot a histogram of the resulting bootstrap sample means.
- Use the bootstrap distribution to construct a 99% confidence interval for the mean.

**Problem 4.9**
Using the bootstrap sampling distribution of the mean generated in Problem 4.8 (from the first 100 data points in `prob4.csv`), generate 1000 confidence intervals for the sample mean, each based on a group of n=4 randomly sampled values from the full dataset in `prob4.csv`.

For each confidence interval:
- Use the bootstrap distribution to determine the interval bounds (for example, the 0.5th and 99.5th percentiles of the bootstrap means).

What percentage of these intervals contain the true population mean (computed from all data in `prob4.csv`)?

**Problem 4.10**
Is 100 samples enough for reasonable bootstrapping? Compare your confidence interval to that obtained with the full dataset (Problems 4.6–4.7). Discuss any limitations or potential biases that may arise with only 100 samples.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

S = pd.read_csv('/Workspace/Users/rngualli@go.olemiss.edu/Homework4/DataScienceOlemiss/prob4.csv', header=0).iloc[:,0].values

S_100 = S[:100]
pop_mean = np.mean(S)
n = 100
num_bootstrap = 10000

# Problem 4.8 - Bootstrap from first 100 points
boot_means = [np.mean(np.random.choice(S_100, n, replace=True)) for _ in range(num_bootstrap)]

plt.hist(boot_means, bins=30, edgecolor='black', density=True)
plt.xlabel('Bootstrap Sample Mean')
plt.ylabel('Relative Frequency')
plt.title('Bootstrap Sampling Distribution (n=100, first 100 points)')
plt.grid(True, alpha=0.3)
plt.show()

low, high = np.percentile(boot_means, 0.5), np.percentile(boot_means, 99.5)
print(f"Bootstrap 99% CI: ({low:.4f}, {high:.4f})")

# Problem 4.9 - Use bootstrap CI bounds on n=4 groups from full dataset
n_group = 4
num_groups = 1000
contains = sum(low <= np.mean(np.random.choice(S, n_group, replace=False)) <= high for _ in range(num_groups))
print(f"Coverage on full dataset (n=4): {contains/num_groups:.4f}")

# Problem 4.10 - Compare
print(f"\nTrue population mean:            {pop_mean:.4f}")
print(f"Mean of first 100 points:        {np.mean(S_100):.4f}")
print(f"Bootstrap CI from 100 points:    ({low:.4f}, {high:.4f})")